# 00bis — Nouvelles photos de jardins : annotations Gemini Vision → PostgreSQL

**Objectif :** parcourir `ai-service/plant-ingestion/inputs/images`, détecter uniquement les photos nouvelles (SHA-256), obtenir les neuf indicateurs Gemini Vision, et persister les résultats **non revus** dans PostgreSQL.

- Les photos ne sont pas envoyées à Gemini tant que `RUN_GEMINI=True` n'est pas défini.
- Les annotations automatiques restent `reviewed=false` et les photos `PENDING` (ou statut existant si déjà connues). Le notebook ne crée **jamais** de vérité terrain `GARDEN_TYPE`.
- Les photos sont envoyées à une API externe : n'utiliser que des photos pour lesquelles tu disposes des droits et du consentement approprié.
- Pas de CSV obligatoire : le dossier images est la source; SHA-256 évite les doublons.
- Pré-requis : tables Flyway `horticulture.garden_image_candidate` et `horticulture.ml_garden_features` déjà présentes.



In [ ]:
from pathlib import Path
import os,hashlib,json,base64,time,mimetypes
from urllib.request import Request,urlopen
from urllib.error import HTTPError
from dotenv import load_dotenv
import psycopg
from psycopg import sql
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"ai-service/plant-ingestion/inputs/images").is_dir())
load_dotenv(ROOT/".env",override=False)
load_dotenv(ROOT/".env.local",override=False)
IMAGES=ROOT/"ai-service/plant-ingestion/inputs/images"
DSN=os.environ["GARDNAIZE_ML_DATABASE_URL"].replace("postgresql+psycopg://","postgresql://",1)
FEATURES=("lawn","terrace","pathway","trees","shrubs","fence","low_wall","stairs","slope")
MODEL=os.getenv("GEMINI_MODEL","gemini-2.5-flash")
RUN_GEMINI=False  # passer à True seulement après vérification des images et consentements
SAVE_TO_DB=False  # passer à True après inspection des réponses
MAX_IMAGES=100
ALLOWED={".jpg",".jpeg",".png",".webp"}
def sha256(path):
 h=hashlib.sha256()
 with path.open("rb") as f:
  for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
 return h.hexdigest()
files=sorted(p for p in IMAGES.rglob("*") if p.is_file() and p.suffix.lower() in ALLOWED)
inventory=pd.DataFrame([{"path":str(p.relative_to(ROOT/"ai-service/plant-ingestion")),"sha256":sha256(p),"size_mib":round(p.stat().st_size/1048576,2)} for p in files])
print("Images trouvées :",len(inventory))
display(inventory.tail(20))



In [ ]:
def table_columns(conn,table):
 with conn.cursor() as cur:
  cur.execute("""SELECT column_name,is_nullable,column_default,data_type
      FROM information_schema.columns WHERE table_schema='horticulture'
      AND table_name=%s ORDER BY ordinal_position""",(table,))
  return {r[0]:{"nullable":r[1]=="YES","default":r[2],"type":r[3]} for r in cur.fetchall()}
with psycopg.connect(DSN) as conn:
 image_cols=table_columns(conn,"garden_image_candidate")
 feature_cols=table_columns(conn,"ml_garden_features")
 assert image_cols and feature_cols,"Migrations ML des jardins absentes"
 assert {"image_id","image_uri"}.issubset(image_cols),"Schéma image inattendu"
 assert {"image_id","feature_values"}.issubset(feature_cols),"Schéma features inattendu"
 with conn.cursor() as cur:
  cur.execute("SELECT image_id,image_uri FROM horticulture.garden_image_candidate")
  known=cur.fetchall()
known_paths={r[1] for r in known}
# Comparaison SHA-256 avec les fichiers locaux référencés : protège contre renommage de photos
known_hashes=set()
for _,uri in known:
 p=(ROOT/"ai-service/plant-ingestion"/uri).resolve()
 if p.is_relative_to(IMAGES.resolve()) and p.is_file():known_hashes.add(sha256(p))
new=[p for p in files if str(p.relative_to(ROOT/"ai-service/plant-ingestion")) not in known_paths and sha256(p) not in known_hashes]
print("Images déjà référencées :",len(known),"| Nouvelles images :",len(new))
print("Colonnes images :",list(image_cols))
print("Colonnes features :",list(feature_cols))
display(pd.DataFrame({"new_image":[str(p.relative_to(IMAGES)) for p in new[:MAX_IMAGES]]}))



In [ ]:
PROMPT="""Analyse cette photographie de jardin. Retourne UNIQUEMENT un objet JSON
avec exactement les neuf clés lawn, terrace, pathway, trees, shrubs, fence,
low_wall, stairs, slope. Pour chaque clé : true si visible, false si absent,
null si impossible à déterminer. Une terrasse comprend terrasse bois,
pierre, béton ou carrelage. Ne déduis PAS la surface dominante ni une
catégorie LAWN_DOMINANT/TERRACE_DOMINANT/MIXED. Ne renvoie aucune autre clé."""
def analyze(path):
 api_key=os.environ.get("GEMINI_API_KEY")
 if not api_key:raise RuntimeError("GEMINI_API_KEY manquante")
 mime=mimetypes.guess_type(path.name)[0] or "image/jpeg"
 payload={"contents":[{"parts":[{"text":PROMPT},
   {"inline_data":{"mime_type":mime,"data":base64.b64encode(path.read_bytes()).decode("ascii")}}]}],
   "generationConfig":{"responseMimeType":"application/json","temperature":0}}
 url=f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
 req=Request(url,data=json.dumps(payload).encode(),headers={"Content-Type":"application/json","x-goog-api-key":api_key},method="POST")
 with urlopen(req,timeout=90) as response:result=json.load(response)
 raw=result["candidates"][0]["content"]["parts"][0]["text"]
 obj=json.loads(raw)
 if set(obj)!=set(FEATURES):raise ValueError(f"Clés Gemini inattendues : {set(obj)^set(FEATURES)}")
 if any(v is not None and not isinstance(v,bool) for v in obj.values()):
  raise ValueError("Gemini doit retourner bool/null uniquement")
 return obj
annotations=[]
if RUN_GEMINI:
 for path in new[:MAX_IMAGES]:
  try:
   values=analyze(path)
   annotations.append({"path":path,"uri":str(path.relative_to(ROOT/"ai-service/plant-ingestion")),
                       "sha256":sha256(path),"features":values})
   print("OK",path.name,values)
  except Exception as e:
   print("ERREUR",path.name,type(e).__name__,str(e)[:300])
  time.sleep(0.4)
else:print("Simulation : RUN_GEMINI=False, aucun appel API.")
print("Annotations obtenues :",len(annotations))



In [ ]:
# Écriture volontairement conservatrice : pas de modification des anciennes annotations.
# Les colonnes obligatoires supplémentaires sont affichées avant toute tentative.
def mandatory_unknown(columns,provided):
 return [name for name,meta in columns.items() if name not in provided
         and not meta["nullable"] and meta["default"] is None]
def persist_one(conn,item):
 img={"image_uri":item["uri"]}
 if "review_status" in image_cols:img["review_status"]="PENDING"
 if "dataset_split" in image_cols:img["dataset_split"]="TRAIN"
 if "sha256" in image_cols:img["sha256"]=item["sha256"]
 if "image_sha256" in image_cols:img["image_sha256"]=item["sha256"]
 # Add source metadata only where the schema supports it.
 if "source_key" in image_cols:img["source_key"]="LOCAL_GARDEN_PHOTOS"
 if "source" in image_cols:img["source"]="LOCAL_GARDEN_PHOTOS"
 required=mandatory_unknown(image_cols,set(img)|{"image_id"})
 if required:raise RuntimeError(f"Colonnes image obligatoires à renseigner selon migration : {required}")
 with conn.cursor() as cur:
  cur.execute(sql.SQL("INSERT INTO horticulture.garden_image_candidate ({}) VALUES ({}) RETURNING image_id").format(
    sql.SQL(",").join(map(sql.Identifier,img)),
    sql.SQL(",").join([sql.Placeholder()]*len(img))),list(img.values()))
  image_id=cur.fetchone()[0]
  values={"image_id":image_id,"feature_values":json.dumps(item["features"])}
  if "reviewed" in feature_cols:values["reviewed"]=False
  if "model_name" in feature_cols:values["model_name"]=MODEL
  if "model" in feature_cols:values["model"]=MODEL
  if "provider" in feature_cols:values["provider"]="GEMINI"
  if "source" in feature_cols:values["source"]="GEMINI"
  required=mandatory_unknown(feature_cols,set(values))
  if required:raise RuntimeError(f"Colonnes features obligatoires à renseigner : {required}")
  cur.execute(sql.SQL("INSERT INTO horticulture.ml_garden_features ({}) VALUES ({})").format(
    sql.SQL(",").join(map(sql.Identifier,values)),
    sql.SQL(",").join([sql.Placeholder()]*len(values))),list(values.values()))
 return image_id
if SAVE_TO_DB:
 assert RUN_GEMINI and annotations,"Aucune annotation Gemini à persister"
 inserted=[]
 for item in annotations:
  try:
   with psycopg.connect(DSN) as conn:
    # Recheck path in transaction; rerunning won't overwrite old data
    with conn.cursor() as cur:
     cur.execute("SELECT 1 FROM horticulture.garden_image_candidate WHERE image_uri=%s",(item["uri"],))
     exists=cur.fetchone()
    if exists:
     print("Déjà en base :",item["uri"]);continue
    image_id=persist_one(conn,item)
    inserted.append((image_id,item["uri"]))
  except Exception as exc:
   print("Insertion échouée :",item["uri"],type(exc).__name__,str(exc))
 print("Nouvelles images + annotations insérées :",len(inserted))
else:print("Aucune écriture : SAVE_TO_DB=False")



In [ ]:
with psycopg.connect(DSN) as conn:
 with conn.cursor() as cur:
  cur.execute("""SELECT g.image_id,g.image_uri,g.review_status,
     f.reviewed,f.feature_values
     FROM horticulture.garden_image_candidate g
     JOIN horticulture.ml_garden_features f USING(image_id)
     ORDER BY g.image_id DESC LIMIT 30""")
  recent=cur.fetchall()
display(pd.DataFrame(recent,columns=["image_id","image_uri","review_status","features_reviewed","features"]))
print("Étape suivante : revue humaine des neuf indicateurs puis étiquetage indépendant du type de jardin dans 00_Jardins_Etiquetage_FR_EN.ipynb")

